# CLIP4CAD-GFA v4.4 Training

**Topology-Aware Multimodal Alignment**

## Key Innovations from v4
1. **Topology Message Passing** - Face↔Edge information exchange via edge_to_faces graph
2. **Hierarchical Pattern Aggregation** - Group faces by BFS level for semantic patterns
3. **Multi-Source Query Generation** - Attend to tokens + level summaries + global summary
4. **Contrastive Query Loss** - Instance-level Q_self → T_feat matching

## Training Stages
- **Stage 1 (Epochs 1-15)**: Foundation grounding with curriculum (90%→15% hints)
- **Stage 2 (Epochs 16-35)**: Alignment refinement with hard negatives

## Loss Weights
| Stage | λ_self | λ_query_con | λ_query_cos | λ_embed | λ_grounding | λ_detail |
|-------|--------|-------------|-------------|---------|-------------|----------|
| 1 | 0.1 | **1.5** | 0.3 | 0.3 | 0.5 | 0.0 |
| 2 | 0.3 | **1.0** | 0.3 | 0.3 | 0.3 | 0.3 |

## Expected Results
- Self-cos BRep: **0.65-0.80** (vs 0.18 in v4)
- Query alignment: **0.85+**
- Text→BRep R@1 (self): **40-60%**

In [ ]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm
import numpy as np
import h5py
from pathlib import Path

# Clean memory
gc.collect()
torch.cuda.empty_cache()

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Configuration
# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS
# ═══════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"
ALIGNED_DIR = DATA_ROOT / "aligned"

# PC and Text files (existing)
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

# Output
OUTPUT_DIR = Path("../outputs/gfa_v4_4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 128  # Smaller due to topology encoder
STAGE1_EPOCHS = 15
STAGE2_EPOCHS = 20
STAGE1_LR = 5e-5
STAGE2_LR = 1e-5
WARMUP_EPOCHS = 3
MAX_GRAD_NORM = 1.0

# Stage 1 loss weights (focus on alignment)
STAGE1_WEIGHTS = {
    'lambda_self': 0.1,
    'lambda_query_contrastive': 1.5,
    'lambda_query_cosine': 0.3,
    'lambda_embed': 0.3,
    'lambda_grounding': 0.5,
    'lambda_detail': 0.0,
}

# Stage 2 loss weights (add detail discrimination)
STAGE2_WEIGHTS = {
    'lambda_self': 0.3,
    'lambda_query_contrastive': 1.0,
    'lambda_query_cosine': 0.3,
    'lambda_embed': 0.3,
    'lambda_grounding': 0.3,
    'lambda_detail': 0.3,
}

print("Configuration:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Stage 1: {STAGE1_EPOCHS} epochs @ LR={STAGE1_LR}")
print(f"  Stage 2: {STAGE2_EPOCHS} epochs @ LR={STAGE2_LR}")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
# Cell 3: Import Dataset
# Use existing GFAMappedDataset with AutoBrep topology support
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

print("Using GFAMappedDataset with use_autobrep=True")
print("This loads:")
print("  - BRep features with topology (edge_to_faces, bfs_level, face_centroids)")
print("  - PC features from ShapeLLM")
print("  - Text features from Phi-4")
print()
print("Required files:")
print(f"  - {EMBEDDINGS_DIR / 'train_brep_autobrep.h5'}")
print(f"  - {EMBEDDINGS_DIR / 'val_brep_autobrep.h5'}")
print(f"  - {ALIGNED_DIR / 'uid_mapping_autobrep.json'}")
print(f"  - {PC_FILE}")
print(f"  - Text embeddings (pre-split or full)")

In [ ]:
# Cell 4: Load Datasets
gc.collect()
torch.cuda.empty_cache()

print("Loading datasets with AutoBrep topology...")
print("=" * 60)

# Limit to 111k samples to fit text embeddings in RAM
MAX_TRAIN_SAMPLES = 111000

# Train dataset - LOAD TO RAM for fast training
print(f"\n[1/2] Loading TRAIN dataset (max {MAX_TRAIN_SAMPLES:,} samples)...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=True,
    use_live_text=False,
    use_autobrep=True,  # Enable topology fields
    autobrep_dir=str(EMBEDDINGS_DIR),
    max_samples=MAX_TRAIN_SAMPLES,
)
print(f"Train: {len(train_dataset):,} samples")

# Val dataset - ON DISK (saves RAM)
print("\n[2/2] Loading VAL dataset...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=False,  # Save RAM
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
)
print(f"Val: {len(val_dataset):,} samples")

print("\n" + "=" * 60)
print("DATASETS READY!")

In [ ]:
# Cell 5: Create DataLoaders

# Create dataloaders with existing collate function
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

# Verify sample has topology fields
sample = train_dataset[0]
print(f"\nSample keys: {list(sample.keys())}")
if 'edge_to_faces' in sample:
    print(f"  edge_to_faces: {sample['edge_to_faces'].shape}")
    print(f"  bfs_level: {sample['bfs_level'].shape}")
    print(f"  face_centroids: {sample['face_centroids'].shape}")
else:
    print("  WARNING: Topology fields not found!")

In [ ]:
# Cell 6: Create Model
gc.collect()
torch.cuda.empty_cache()

from clip4cad.models.clip4cad_gfa_v4_4 import CLIP4CAD_GFA_v44, GFAv44Config
from clip4cad.losses.gfa_v4_4_losses import (
    GFAv44Loss, 
    compute_self_grounding_quality, 
    compute_query_alignment
)

# Model configuration
config = GFAv44Config(
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    d_unified=256,
    d_proj=128,
    d_ground=128,
    num_slots=12,
    num_detail_queries=8,
    num_heads=8,
    # Topology encoder
    num_msg_layers=3,
    num_brep_tf_layers=4,
    max_bfs_levels=32,
    # Pattern aggregator
    num_pattern_levels=10,
    # Query generator
    num_query_layers=4,
    num_pc_query_layers=2,
    dropout=0.1,
)

model = CLIP4CAD_GFA_v44(config).to(device)
print(f"Model parameters: {model.count_parameters():,}")
print(f"Trainable parameters: {model.count_parameters(trainable_only=True):,}")

In [ ]:
# Cell 7: Training Setup

# Loss
criterion = GFAv44Loss(**STAGE1_WEIGHTS)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=STAGE1_LR,
    weight_decay=0.01,
    betas=(0.9, 0.999),
)

# Scaler for FP16
scaler = GradScaler()

# Scheduler
total_epochs = STAGE1_EPOCHS + STAGE2_EPOCHS
warmup_steps = WARMUP_EPOCHS * len(train_loader)
total_steps = total_epochs * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(1e-6 / STAGE1_LR, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)

# ═══════════════════════════════════════════════════════════════════════════════
# BATCH KEY REMAPPING (collate outputs brep_* but model expects *)
# ═══════════════════════════════════════════════════════════════════════════════
def remap_batch(batch):
    """Remap collate output keys to model expected keys."""
    remapped = {}
    for k, v in batch.items():
        # Remove 'brep_' prefix for model compatibility
        if k.startswith('brep_'):
            new_key = k[5:]  # Remove 'brep_' prefix
            remapped[new_key] = v
        else:
            remapped[k] = v
    return remapped

# ═══════════════════════════════════════════════════════════════════════════════
# CURRICULUM SCHEDULE
# ═══════════════════════════════════════════════════════════════════════════════

def get_cond_dropout(epoch, stage):
    """Smoother curriculum - gradual hint removal."""
    if stage == 2:
        stage2_epoch = epoch - STAGE1_EPOCHS
        if stage2_epoch <= 3:
            return 0.85   # 15% hints
        elif stage2_epoch <= 6:
            return 0.92   # 8% hints
        elif stage2_epoch <= 10:
            return 0.97   # 3% hints
        else:
            return 1.0    # 0% hints
    
    # Stage 1
    if epoch <= 2:
        return 0.1   # 90% hints
    elif epoch <= 4:
        return 0.2   # 80% hints
    elif epoch <= 6:
        return 0.35  # 65% hints
    elif epoch <= 8:
        return 0.5   # 50% hints
    elif epoch <= 10:
        return 0.65  # 35% hints
    elif epoch <= 12:
        return 0.75  # 25% hints
    else:
        return 0.85  # 15% hints

print("Training setup complete!")

In [ ]:
# Cell 8: Training Loop

global_step = 0
best_val_loss = float('inf')
best_self_cosine = 0.0
current_stage = 1

history = {
    'train_loss': [], 'val_loss': [],
    'self_cosine_brep': [], 'self_cosine_pc': [],
    'query_align_brep': [], 'query_align_pc': [],
}

print("Starting GFA v4.4 Training...")
print("=" * 70)

for epoch in range(1, total_epochs + 1):
    
    # ─────────────────────────────────────────────────────────────────────────
    # STAGE TRANSITION
    # ─────────────────────────────────────────────────────────────────────────
    if epoch == STAGE1_EPOCHS + 1:
        print("\n" + "=" * 70)
        print("TRANSITIONING TO STAGE 2")
        print("=" * 70)
        current_stage = 2
        
        criterion.update_weights(**STAGE2_WEIGHTS)
        print(f"Updated loss weights")
        
        for param_group in optimizer.param_groups:
            param_group['lr'] = STAGE2_LR
        print(f"Reduced LR to {STAGE2_LR}")
        
        # Save Stage 1 checkpoint
        torch.save({
            'epoch': epoch - 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_self_cosine': best_self_cosine,
        }, OUTPUT_DIR / 'checkpoint_stage1_final.pt')
    
    # ─────────────────────────────────────────────────────────────────────────
    # UPDATE CURRICULUM
    # ─────────────────────────────────────────────────────────────────────────
    cond_drop = get_cond_dropout(epoch, current_stage)
    model.set_cond_dropout(cond_drop, cond_drop + 0.1)  # PC slightly higher
    hint_pct = int((1 - cond_drop) * 100)
    
    # ─────────────────────────────────────────────────────────────────────────
    # TRAIN EPOCH
    # ─────────────────────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {
        'self_cos_brep': [], 'self_cos_pc': [],
        'query_align_brep': [], 'query_align_pc': []
    }
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} (S{current_stage}, {hint_pct}% hints)")
    
    for batch_idx, batch in enumerate(pbar):
        batch = remap_batch(batch)  # Convert brep_* keys to model expected keys
        
        with autocast():
            outputs = model(batch)
            loss, loss_dict = criterion(outputs, stage=current_stage)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        global_step += 1
        epoch_loss += loss_dict['total'].item()
        
        # Compute metrics
        with torch.no_grad():
            if 'z_brep' in outputs:
                cos_brep = compute_self_grounding_quality(
                    outputs['z_brep'], outputs['z_brep_self']
                )
                epoch_metrics['self_cos_brep'].append(cos_brep)
            
            if 'z_pc' in outputs:
                cos_pc = compute_self_grounding_quality(
                    outputs['z_pc'], outputs['z_pc_self']
                )
                epoch_metrics['self_cos_pc'].append(cos_pc)
            
            if 'Q_brep_self' in outputs:
                q_align_brep = compute_query_alignment(
                    outputs['Q_brep_self'], outputs['T_feat'], outputs['confidence']
                )
                epoch_metrics['query_align_brep'].append(q_align_brep)
        
        # Progress bar
        postfix = {
            'loss': f"{loss_dict['total'].item():.3f}",
            'guided': f"{loss_dict['guided'].item():.3f}",
            'q_con': f"{loss_dict['query_con'].item():.3f}",
            'cos': f"{epoch_metrics['self_cos_brep'][-1]:.3f}" if epoch_metrics['self_cos_brep'] else "N/A",
        }
        pbar.set_postfix(postfix)
    
    # ─────────────────────────────────────────────────────────────────────────
    # EPOCH SUMMARY
    # ─────────────────────────────────────────────────────────────────────────
    avg_loss = epoch_loss / len(train_loader)
    avg_cos_brep = np.mean(epoch_metrics['self_cos_brep']) if epoch_metrics['self_cos_brep'] else 0
    avg_cos_pc = np.mean(epoch_metrics['self_cos_pc']) if epoch_metrics['self_cos_pc'] else 0
    avg_q_brep = np.mean(epoch_metrics['query_align_brep']) if epoch_metrics['query_align_brep'] else 0
    
    history['train_loss'].append(avg_loss)
    history['self_cosine_brep'].append(avg_cos_brep)
    history['self_cosine_pc'].append(avg_cos_pc)
    history['query_align_brep'].append(avg_q_brep)
    
    if avg_cos_brep > best_self_cosine:
        best_self_cosine = avg_cos_brep
    
    print(f"\nEpoch {epoch}: Loss={avg_loss:.4f}")
    print(f"  Self-cos: BRep={avg_cos_brep:.4f}, PC={avg_cos_pc:.4f}")
    print(f"  Query-align BRep: {avg_q_brep:.4f}")
    print(f"  Best self-cosine: {best_self_cosine:.4f}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # VALIDATION (every 5 epochs)
    # ─────────────────────────────────────────────────────────────────────────
    if epoch % 5 == 0:
        model.eval()
        val_loss = 0.0
        val_metrics = {'self_cos_brep': [], 'query_align_brep': []}
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                batch = remap_batch(batch)  # Convert brep_* keys to model expected keys
                with autocast():
                    outputs = model(batch)
                    loss, loss_dict = criterion(outputs, stage=current_stage)
                val_loss += loss_dict['total'].item()
                
                cos_brep = compute_self_grounding_quality(
                    outputs['z_brep'], outputs['z_brep_self']
                )
                val_metrics['self_cos_brep'].append(cos_brep)
                
                q_align = compute_query_alignment(
                    outputs['Q_brep_self'], outputs['T_feat'], outputs['confidence']
                )
                val_metrics['query_align_brep'].append(q_align)
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_cos = np.mean(val_metrics['self_cos_brep'])
        avg_val_q = np.mean(val_metrics['query_align_brep'])
        
        history['val_loss'].append(avg_val_loss)
        print(f"Validation: Loss={avg_val_loss:.4f}, Self-cos={avg_val_cos:.4f}, Q-align={avg_val_q:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_val_loss': best_val_loss,
                'best_self_cosine': best_self_cosine,
            }, OUTPUT_DIR / 'checkpoint_best.pt')
            print("Saved best model!")
    
    # Save periodic checkpoint
    if epoch % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_self_cosine': best_self_cosine,
            'history': history,
        }, OUTPUT_DIR / f'checkpoint_epoch{epoch}.pt')

print("\n" + "=" * 70)
print("Training Complete!")
print(f"Best self-cosine: {best_self_cosine:.4f}")

In [ ]:
# Cell 9: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train')
if history['val_loss']:
    val_epochs = list(range(5, len(history['train_loss']) + 1, 5))
    axes[0, 0].plot(val_epochs, history['val_loss'], 'o-', label='Val')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Self-cosine
axes[0, 1].plot(history['self_cosine_brep'], label='BRep')
axes[0, 1].plot(history['self_cosine_pc'], label='PC')
axes[0, 1].axhline(y=0.8, color='g', linestyle='--', alpha=0.5, label='Target')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Cosine Similarity')
axes[0, 1].set_title('Self-Grounding Quality')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Query alignment
axes[1, 0].plot(history['query_align_brep'])
axes[1, 0].axhline(y=0.85, color='g', linestyle='--', alpha=0.5, label='Target')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Query Alignment')
axes[1, 0].set_title('BRep Query Alignment')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Stage transition
for ax in axes.flat:
    ax.axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', alpha=0.3, label='Stage 2')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

In [ ]:
# Cell 10: Save Final Model
torch.save({
    'config': config,
    'model_state_dict': model.state_dict(),
    'best_self_cosine': best_self_cosine,
    'history': history,
}, OUTPUT_DIR / 'gfa_v4_4_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'gfa_v4_4_final.pt'}")
print(f"Best self-cosine: {best_self_cosine:.4f}")